# 00 — Orientation: From Theory to Practice

You've been through the theory phase (`0-tutorial/`) — you can now say what retrieval, chunking, faithfulness, and correctness each mean, and why "the answer was wrong" is never a complete sentence on its own. This notebook, and the eleven after it, are where that vocabulary gets used on real, messy data instead of clean illustrative examples.

The core principle for this whole series, unchanged from the plan: **never learn a metric first — become the metric, then learn how a framework automates it.**


## Why real, borrowed datasets instead of StoryVerse

The StoryVerse series worked because 5 short stories made every concept visible at a glance. Evaluation needs the opposite: real documents, real ambiguity, real "the answer sounds right but isn't." So this series pulls from four open, public datasets instead — chosen so every question type you'll practice on has a natural home:

| Dataset | What it's for here | Why it fits |
|---|---|---|
| **CUAD** (contract QA) | Single-fact lookup, numeric/date extraction | Real commercial contracts, labeled by clause type |
| **SQuAD 2.0** | Unanswerable questions | Half its validation set is deliberately unanswerable from the given context |
| **HotpotQA** | Multi-document comparison | Questions that need two separate passages combined |
| **MS MARCO** | Retrieval metrics | Ships with real ground-truth relevance labels already attached |

Every tutorial in this series pulls a **fixed, reproducible slice** of these — same examples every run, nothing for you to author. Your only job across this series is the judgment itself, not data collection.


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)


## Loading the fixed 20-example set

Twenty examples, stratified across four question types: 5 single-fact lookups, 5 numeric extractions, 5 multi-document comparisons, 5 unanswerable questions. This is the exact same 20 you'll use in every notebook through `09`.

Two adjustments from how these datasets are sometimes described elsewhere, found by actually loading them: CUAD's original HF repo (`theatticusproject/cuad-qa`) uses a legacy "dataset script" format the current `datasets` library no longer loads at all -- `chenghao/cuad_qa` (its `test` split) is a modern-format mirror with the identical clause-category structure. And this version of SQuAD 2.0 has no `is_impossible` field -- an unanswerable question here is simply one with an empty `answers.text` list.


In [ ]:
from datasets import load_dataset

cuad = load_dataset("chenghao/cuad_qa", split="test", streaming=True)
hotpot = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation", streaming=True)
squad2 = load_dataset("rajpurkar/squad_v2", split="validation", streaming=True)


def first_n_matching(dataset, predicate, n, scan_limit=5000):
    """Stream through the dataset and return the first n examples matching predicate."""
    out = []
    for i, ex in enumerate(dataset):
        if predicate(ex):
            out.append(ex)
        if len(out) == n or i >= scan_limit:
            break
    return out


governing_law = first_n_matching(cuad, lambda ex: ex["question"] == "Governing Law", 5)
cap_on_liability = first_n_matching(cuad, lambda ex: ex["question"] == "Cap On Liability", 5)
comparisons = first_n_matching(hotpot, lambda ex: ex["type"] == "comparison", 5)
unanswerable = first_n_matching(squad2, lambda ex: len(ex["answers"]["text"]) == 0, 5)

print(f"Governing Law:    {len(governing_law)} found")
print(f"Cap On Liability: {len(cap_on_liability)} found")
print(f"Comparisons:      {len(comparisons)} found")
print(f"Unanswerable:     {len(unanswerable)} found")


## Normalizing four different shapes into one

CUAD, HotpotQA, and SQuAD2 don't share a schema. Before anything else can happen, they need one common shape: `question`, `context_text`, `gold_answer`, `source`.

One more real constraint, found by actually trying to send these to the model: full CUAD contracts run up to **175,000 characters** -- real commercial contracts are long, and a couple of these blow straight through this account's rate limit in a single request (a genuine `413 Request too large` from Groq). This is worth sitting with rather than just patching around: no real RAG system hands an LLM an entire 175,000-character document either. A retriever would have found the relevant chunk first. So rather than truncate arbitrarily, extract a window around where the labeled answer actually sits -- roughly what a working retrieval step would have handed the generator anyway.


In [ ]:
def flatten_hotpot_context(ex: dict) -> str:
    parts = []
    for title, sentences in zip(ex["context"]["title"], ex["context"]["sentences"]):
        parts.append(f"{title}: " + " ".join(sentences))
    return "\n".join(parts)


def window_around_answer(context: str, answer_start: int, window: int = 700) -> str:
    """Simulate a retrieved chunk: a window of text around the labeled answer span,
    instead of the entire source document."""
    start = max(0, answer_start - window)
    end = min(len(context), answer_start + window)
    return context[start:end]


def normalize(ex: dict, source: str) -> dict:
    if source == "cuad":
        has_answer = bool(ex["answers"]["text"])
        gold = ex["answers"]["text"][0] if has_answer else "Not specified in the contract."
        context_text = (
            window_around_answer(ex["context"], ex["answers"]["answer_start"][0])
            if has_answer
            else ex["context"][:1400]
        )
    elif source == "hotpot":
        context_text = flatten_hotpot_context(ex)
        gold = ex["answer"]
    elif source == "squad2":
        context_text = ex["context"]
        gold = "Not stated in the documents."
    return {"question": ex["question"], "context_text": context_text, "gold_answer": gold, "source": source}


examples = (
    [normalize(ex, "cuad") for ex in governing_law]        # indices 0-4:   single-fact
    + [normalize(ex, "cuad") for ex in cap_on_liability]    # indices 5-9:   numeric
    + [normalize(ex, "hotpot") for ex in comparisons]        # indices 10-14: multi-doc
    + [normalize(ex, "squad2") for ex in unanswerable]       # indices 15-19: unanswerable
)

print(f"Total examples: {len(examples)}")
print(f"[0] ({examples[0]['source']}) {examples[0]['question']}")
print(f"[15] ({examples[15]['source']}) {examples[15]['question']}")
print()
context_lengths = [len(ex["context_text"]) for ex in examples]
print(f"Context sizes: min {min(context_lengths)}, max {max(context_lengths)} chars")


## Generating the "system under test"

Every notebook from here on needs something to actually evaluate. Rather than bring your own RAG system, have the model play that role directly: answer strictly from the given context, exactly like a grounded RAG pipeline would. For the 5 unanswerable examples, also generate a deliberately **overconfident** version -- an answer that fabricates something plausible instead of admitting the context doesn't cover it. You'll need both versions later: the honest one to practice recognizing "this is actually fine," and the overconfident one to practice catching unfaithfulness that *sounds* completely reasonable.


In [ ]:
def generate_answer(question: str, context: str) -> str:
    prompt = (
        f"Context:\n{context}\n\nQuestion: {question}\n\n"
        "Answer using only the context above. If the context does not contain the answer, say so explicitly."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


def generate_overconfident_answer(question: str, context: str) -> str:
    prompt = (
        f"Context:\n{context}\n\nQuestion: {question}\n\n"
        "Answer the question directly and confidently, even if you have to infer or extrapolate beyond what's explicitly stated."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


for ex in examples:
    ex["system_answer"] = generate_answer(ex["question"], ex["context_text"])

for ex in examples[15:20]:
    ex["system_answer_overconfident"] = generate_overconfident_answer(ex["question"], ex["context_text"])

print("Done generating answers for all 20 examples, plus 5 overconfident variants.")


In [ ]:
print("Example [0] -- single-fact lookup:")
print("  Q:", examples[0]["question"])
print("  Gold:", examples[0]["gold_answer"])
print("  System:", examples[0]["system_answer"][:200])
print()
print("Example [15] -- unanswerable:")
print("  Q:", examples[15]["question"])
print("  Honest system answer:", examples[15]["system_answer"][:200])
print("  Overconfident variant:", examples[15]["system_answer_overconfident"][:200])


Look closely at the two versions of example 15's answer. Sometimes the honest one correctly says "the documents don't address this," and the overconfident one invents a specific-sounding claim on top. But don't be surprised if, for some of the 5 unanswerable examples, *both* versions just answer the question anyway -- SQuAD 2.0's unanswerable questions are deliberately adversarial: worded to look answerable from the same passage, with a subtle mismatch (a date range that's off by a few years, a name that's close but not quite what's asked about) that a fast reader -- human or model -- glides straight past. If that happens here, that's not a broken example. It's the first real preview of this series' whole theme: "wrong" is never one thing. Sometimes the system missed something real. Sometimes the question itself was quietly harder than it looked.


Keep an eye on both versions as you move through the series -- whichever way each of your 5 came out, they're the clearest example you'll have of the exact same question producing a faithful answer and an unfaithful one, from the exact same model, with only the instruction changed.


## Saving the fixed set

Everything from here on -- notebooks `01` through `09` -- loads this file instead of regenerating it. Same reason `embeddings.json` existed in the StoryVerse series: expensive to build once, cheap to reload forever.


In [ ]:
examples_path = os.path.join(DATA_DIR, "eval_examples.json")
with open(examples_path, "w") as f:
    json.dump(examples, f, indent=2)

print(f"Saved {len(examples)} examples to {examples_path}")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Stratified sample | A fixed slice of examples chosen to cover distinct categories on purpose, not randomly |
| Gold answer | The dataset's own labeled correct answer, used as ground truth |
| Overconfident answer | An answer that guesses beyond what the context actually supports |

You now have 20 real question/context/answer sets, covering four distinct failure-prone categories, with zero data to author yourself. Every notebook from here on reuses exactly this set -- when you compare your own judgment against an automated one later, in notebook `07`, you're comparing against the *same* examples, not a different sample that could quietly explain away a disagreement.

**Next up:** notebook `01` puts you in the reviewer's seat -- read all 20, and decide for each one whether the system actually got it right.

**Exercises:**

- Print `examples[10]` in full (the first HotpotQA comparison example) -- can you answer it yourself, using only the printed context, before reading the system's answer?
- Increase `scan_limit` and pull 5 more Governing Law examples (indices 5-9 instead of 0-4) -- do they look meaningfully different in structure?
- Try `generate_overconfident_answer` on one of the single-fact CUAD examples (which *does* have a real answer) -- does a higher temperature change anything when the context actually supports a clear answer?
